# 🤖 ClipNest — Telegram Downloader Bot (Google Colab)

Ye notebook aapke repo se **Telegram-first downloader bot** Colab par chalata hai.
Web UI / tunnel start nahi hota — sirf bot.

### 3 steps
1. `Runtime → Run all` dabayein (ya cells upar se niche tak ek-ek run karein).
2. BotFather ka token paste karein — input masked rehta hai, notebook mein save nahi hota.
3. Telegram mein bot ko `/start` bhejein.

### Dhyan rakhein
- ⏳ Colab **temporary** hai — runtime disconnect/band hote hi bot offline ho jayega.
- 📦 Cloud Bot API par bot **50 MB** se badi file upload nahi kar sakta. Badi files ke liye VPS + local Bot API server chahiye (README dekhein).
- 🔐 Token kisi public notebook, screenshot ya GitHub par save na karein.
- ⚖️ Sirf wahi content download karein jiska aapko haq hai.

> **Tip:** Colab Secrets (🔑 left sidebar) mein `CLIPNEST_TELEGRAM_BOT_TOKEN` save karein — notebook
> automatically wahan se token utha lega aur aapko har baar paste nahi karna padega.


## 1️⃣ Setup — clone + install


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/gintokisakata918-prog/Tg-downloder-.git"
APP_DIR = Path("/content/Tg-downloder-")


def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, cwd=cwd, check=True)


if sys.version_info < (3, 11):
    raise SystemExit("❌ Python 3.11+ chahiye — Colab runtime update karein.")

# FFmpeg video+audio merge aur MP3 conversion ke liye zaroori hai.
if shutil.which("ffmpeg"):
    print("✅ FFmpeg already installed")
else:
    print("📦 FFmpeg install ho raha hai...")
    subprocess.run(
        "apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null 2>&1",
        shell=True,
        check=True,
    )

# Repo clone karein, ya already clone hai to latest main par le aayein.
if (APP_DIR / ".git").is_dir():
    print("🔄 Latest code la raha hoon...")
    run(["git", "fetch", "--depth", "1", "origin", "main"], cwd=APP_DIR)
    run(["git", "reset", "--hard", "FETCH_HEAD"], cwd=APP_DIR)
else:
    print("⬇️ Repo clone ho raha hai...")
    run(["git", "clone", "--depth", "1", REPO, str(APP_DIR)])

print("📦 Dependencies install ho rahi hain...")
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[bot]"], cwd=APP_DIR)
run(
    [sys.executable, "-c", "import telegram, yt_dlp; print('✅ telegram', telegram.__version__, '· yt-dlp', yt_dlp.version.__version__)"],
    cwd=APP_DIR,
)

print(f"\n✅ Setup complete: {APP_DIR}")


## 2️⃣ Token & settings


In [ ]:
#@title 2️⃣ Token & settings { display-mode: "form" }
#@markdown **Private bot?** Apni Telegram user ID daalein (multiple IDs comma se). Khaali = bot sabke liye open. Apni ID bot ko `/id` bhej kar milegi.
ALLOWED_USERS = "" #@param {type:"string"}
#@markdown Maximum video duration (ghante):
MAX_DURATION_HOURS = 2 #@param {type:"slider", min:1, max:6, step:1}
#@markdown Maximum download size (MB). Telegram cloud upload limit 50 MB hai; isliye bahut badi file select karne par upload fail hoga.
MAX_FILE_MB = 100 #@param {type:"integer"}

import getpass
import json
import os
import re
import urllib.request

if "APP_DIR" not in globals():
    raise SystemExit("❌ Pehle upar wala Setup cell chalayein.")


def _colab_secret(name: str) -> str:
    """Colab Secrets se value uthao (agar notebook ko access diya ho)."""
    try:
        from google.colab import userdata

        return (userdata.get(name) or "").strip()
    except Exception:
        return ""


token = _colab_secret("CLIPNEST_TELEGRAM_BOT_TOKEN")
if not token:
    token = getpass.getpass("BotFather BOT TOKEN paste karein (hidden): ").strip()

if not re.fullmatch(r"\d+:[A-Za-z0-9_-]{30,}", token):
    raise SystemExit("❌ Token ka format sahi nahi. @BotFather se /newbot karke token copy karein.")

# Bot shuru karne se pehle Telegram se token confirm karein.
try:
    with urllib.request.urlopen(
        f"https://api.telegram.org/bot{token}/getMe", timeout=20
    ) as response:
        payload = json.load(response)
except Exception as exc:
    raise SystemExit(f"❌ Telegram se connect nahi hua: {exc}") from exc
if not payload.get("ok"):
    raise SystemExit("❌ Token invalid ya revoke ho gaya. @BotFather se naya token lein.")

bot_username = payload["result"]["username"]

os.environ.update(
    {
        "CLIPNEST_TELEGRAM_BOT_TOKEN": token,
        "CLIPNEST_TELEGRAM_ALLOWED_USERS": ALLOWED_USERS.replace(" ", ""),
        "CLIPNEST_DOWNLOAD_DIR": str(APP_DIR / "downloads"),
        "CLIPNEST_MAX_CONCURRENT_DOWNLOADS": "2",
        "CLIPNEST_JOB_TTL_MINUTES": "30",
        "CLIPNEST_MAX_FILESIZE_MB": str(MAX_FILE_MB),
        "CLIPNEST_MAX_DURATION_SECONDS": str(MAX_DURATION_HOURS * 3600),
        # Cloud Bot API ki hard limit — badi files ke liye local Bot API chahiye.
        "CLIPNEST_TELEGRAM_MAX_UPLOAD_MB": "50",
        "CLIPNEST_TELEGRAM_QUALITY_LIMIT": "6",
        "CLIPNEST_RATE_LIMIT_PER_10MIN": "20",
    }
)

print(f"✅ Ready — bot: @{bot_username}")
print(f"   Allowed users: {ALLOWED_USERS.strip() or 'sab (open)'}")
print(f"   Max duration: {MAX_DURATION_HOURS} ghante · Max download: {MAX_FILE_MB} MB")
print("   Token output mein print nahi hua.")


## 🍪 Optional — Instagram cookies

Sirf tab enable karein jab Instagram aapke account se login maange. Ye Telegram token se bilkul alag hai — `API_ID`/`API_HASH` ki zaroorat nahi.


In [ ]:
#@title 🍪 Optional: Instagram cookies (skip kar sakte hain) { display-mode: "form" }
#@markdown Sirf tab chahiye jab Instagram login maange. Browser se Netscape-format `cookies.txt` export karein. Sirf **apni** account ki cookies use karein.
USE_INSTAGRAM_COOKIES = False #@param {type:"boolean"}

if not USE_INSTAGRAM_COOKIES:
    print("⏭ Cookies skip ki gayi.")
elif "APP_DIR" not in globals():
    raise SystemExit("❌ Pehle upar wala Setup cell chalayein.")
else:
    import os

    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        print("Koi file upload nahi hui.")
    elif len(uploaded) != 1:
        print("Sirf ek hi `cookies.txt` upload karein.")
    else:
        name, data = next(iter(uploaded.items()))
        cookie_path = APP_DIR / "data" / "cookies.txt"
        cookie_path.parent.mkdir(parents=True, exist_ok=True)
        cookie_path.write_bytes(data)
        os.environ["CLIPNEST_COOKIES_FILE"] = str(cookie_path)
        print(f"✅ Cookies set: {cookie_path.name} (private rakhein)")


## 3️⃣ Start bot


In [ ]:
import os
import signal
import subprocess
import sys

if not os.environ.get("CLIPNEST_TELEGRAM_BOT_TOKEN"):
    raise SystemExit("❌ Pehle upar wala settings cell chalayein.")


def _stop_stale_bots() -> None:
    """Pichle run ka bot abhi bhi chal raha ho to band karo.

    Warna Telegram "Conflict: terminated by other getUpdates request" dega.
    """
    try:
        result = subprocess.run(
            ["pgrep", "-f", "app.telegram_bot"], capture_output=True, text=True
        )
    except FileNotFoundError:
        return
    for raw in result.stdout.split():
        pid = int(raw)
        if pid == os.getpid():
            continue
        try:
            os.kill(pid, signal.SIGTERM)
            print(f"🧹 Purana bot process band kiya (pid {pid})")
        except ProcessLookupError:
            pass


_stop_stale_bots()
print("🤖 Bot start ho raha hai — Telegram par /start bhejein.", flush=True)
print("⏹ Band karne ke liye cell ke Stop button dabayein.\n", flush=True)

process = subprocess.Popen([sys.executable, "-m", "app.telegram_bot"], cwd=str(APP_DIR))
try:
    process.wait()
except KeyboardInterrupt:
    print("\n🛑 Bot band ho raha hai...")
    process.terminate()
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.kill()
    print("✅ Bot band ho gaya.")


## 🛠️ Troubleshooting

| Problem | Solution |
| --- | --- |
| **Conflict: terminated by other getUpdates request** | Same token kisi aur Colab/server par chal raha hai. Start cell dobara run karein (ye purana process khud band karta hai), ya dusri jagah bot band karein. |
| **Unauthorized / token invalid** | Token galat ya revoke hai. @BotFather se `/mybots → API Token` se dobara copy karein. |
| **File too large / upload fail** | Cloud Bot API ki limit **50 MB** hai. Kam quality chunein, ya VPS par local Bot API server ke saath chalayein. |
| **Bot reply nahi kar raha** | Start cell chal rahi hai na? Colab disconnect ho gaya ho to `Runtime → Reconnect` karke cells dobara run karein. |
| **Instagram login error** | Cookies cell enable karke fresh `cookies.txt` upload karein. |
| **Code update chahiye** | Setup cell dobara run karein (latest `main` pull hota hai), phir settings + start cells. |
| **Disk full** | `!rm -rf /content/Tg-downloder-/downloads/*` chalayein. |

### Bot commands
`/start` · `/help` · `/settings` · `/status` · `/cancel` · `/id` · `/privacy`

### 24×7 chalana hai?
Colab iske liye nahi bana. Hamesha on rakhne ke liye README ke **Docker / VPS** section se deploy karein.
